# Estimate Annual Generation for GM-SEUS Arrays (pvlib + TMY)
NOTE: Requires an environment with pvlib >= 0.12 (NSRDB PSM4 functions), geopandas, scipy, and pvcz (`pip install pvcz`). The BigPanelTilt.yml environment is the closest existing environment.

This script models expected annual AC generation (MWh) for every GM-SEUS photovoltaic array from its installation year to 25 years after installation. Generation is modeled once per array with a Typical Meteorological Year (TMY) and a PVWatts-style _pvlib_ model chain. A temperature-zone performance loss rate (degradation) is then applied for each operating year. Because TMY is the same weather every year, operating year 1 is the undegraded TMY total and every later year differs only by degradation.

**Outputs**
* **GMSEUS_Generation_Annual_Wide**: `arrayID`, `genYr1` ... `genYr25` (MWh). For the GM-SEUS release.
* **GMSEUS_Generation_Annual_Long**: `arrayID`, `instYr`, `opYr`, `calYr`, `MWh`. For joining to other annual time series by calendar year.
* **GMSEUS_Generation_Shape12x24**: `arrayID` and 288 columns (`m01h00` ... `m12h23`) giving the share of a TMY year's energy produced in each month-by-hour cell (rows sum to 1). Used to weight time-varying quantities by when each array actually generates.
* **GMSEUS_Generation_ModelInputs**: the per-array model inputs actually used (mount, tilt, azimuth, axis, GCR, DC:AC ratio, temperature zone, PLR) with provenance columns, for auditing.

**Weather**
* Option: NSRDB PSM4 TMY (NLR Developer Network, free API key) via `pvlib.iotools.get_nsrdb_psm4_tmy`, ~4 km grid. Arrays are snapped to weather cells so each cell is downloaded once and cached.
* Alternative: PVGIS TMY via `pvlib.iotools.get_pvgis_tmy` (no key, no daily cap; same source used for `tiltEst` in script8). PVGIS TMY has no albedo, so 0.2 is used.
* Set with `weatherSource` ('pvgis' default). Cache files, checkpoint, and outputs are named by source, so switching sources later requires no other changes.

**Degradation**
* Jordan, D. C., et al. (2022). Photovoltaic fleet degradation insights. Progress in Photovoltaics, 30, 1166-1175. Median PLR by PV temperature zone (T3: 0.48, T4: 0.78, T5: 0.88 %/year). Mount and module type were not differentiated because Jordan et al. found no statistically significant difference (p > 0.49).
* PV temperature zones from Karin et al. (2019) via the `pvcz` package (rack-mount, 1.1 eV equivalent temperature zone), the same zones used by Jordan et al.

**Key assumptions** (all set as variables in the first code cells)
* Mount: `mixed*` arrays modeled as fixed-axis (conservative). `unknown` arrays assigned by a capacity cutoff derived from arrays with known fixed-axis vs single-axis mounts (balanced accuracy). `dual_axis` modeled as two-axis tracking.
* `capMWDC` and `capMWAC` used as-is for all arrays (`capMWAC` sets inverter clipping).
* Fixed-axis tilt from `tilt` (reported, gap-filled with `tiltEst` in script8). Only arrays newly assigned fixed-axis here (from `unknown`) can lack tilt; those are filled from a latitude regression fitted to fixed-axis `tilt`.
* Azimuth: where `avgAzimuth` is missing, fixed-axis arrays face due south (180) and single-axis trackers have a north-south axis (axis azimuth 180, panels rotating east to west).
* Single-axis tracker axis azimuth from `avgAzimuth` (long-edge direction), horizontal axis, with backtracking at `GCR2`. Fixed-axis row shading is not modeled (PVWatts-style), so GCR only affects trackers.
* `modType` sets the temperature coefficient only.
* Total system losses (soiling, wiring, mismatch, availability, etc.) = `systemLosses` (PVWatts default 14%). Monofacial modules assumed (bifacial gain not modeled).
* Operating year 1 = `instYr`, treated as a full year of operation. Partial installation-year handling is left to downstream users.
* Linear degradation: MWh(opYr) = MWh_TMY x (1 - PLR x (opYr - 1)).
* CSP arrays (`modType` = csp) are not modeled and receive -9999.

# Import Libraries and Variables

In [ ]:
# Import libraries
import os
import time
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import pvlib
import pvcz
from scipy.spatial import cKDTree

# Suppress specific warnings globally (e.g., DeprecationWarning, FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
temp_path = r'F:\BigPanel_Cache'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
derivedTemp_path = os.path.join(derived_path, r'intermediateProducts')
generation_path = os.path.join(derived_path, r'generation')

# Set final GM-SEUS array path (NOTE: update to the final v2.1 array product name)
gmseusArraysPath = os.path.join(derived_path, r'GMSEUS_v2_1\SHP\GMSEUS_Arrays_Final_2026_v2_1.shp')

# Set weather cache folder (one parquet file per weather cell)
weatherCache_path = os.path.join(temp_path, r'Weather_Cache')

# Set weather source: 'pvgis' (PVGIS TMY, no key, no daily cap) or 'nsrdb' (NSRDB PSM4 TMY, free NLR API key, 5,000 requests/day)
# The source is added to every cache file, checkpoint, and output name, so runs with each source can coexist and be compared.
weatherSource = 'pvgis'

# Set output paths (suffixed by weather source)
genWide_path = os.path.join(generation_path, r'GMSEUS_Generation_Annual_Wide_' + weatherSource + '.csv')
genLong_path = os.path.join(generation_path, r'GMSEUS_Generation_Annual_Long_' + weatherSource + '.parquet')
genShape_path = os.path.join(generation_path, r'GMSEUS_Generation_Shape12x24_' + weatherSource + '.parquet')
genInputs_path = os.path.join(generation_path, r'GMSEUS_Generation_ModelInputs_' + weatherSource + '.csv')
genCheckpoint_path = os.path.join(generation_path, r'genCheckpoint_' + weatherSource + '.parquet')

# Make output and cache folders if they do not exist
os.makedirs(generation_path, exist_ok=True)
os.makedirs(weatherCache_path, exist_ok=True)

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Weather settings

# NLR API key and email are read from environment variables so they are not saved in the notebook
nlrApiKey = os.environ.get('NLR_API_KEY', 'DEMO_KEY') # DEMO_KEY is severely rate limited
nlrEmail = os.environ.get('NLR_API_EMAIL', '')

# Weather cell size in degrees (0.1 = ~10 km). Arrays in the same cell share one TMY file; solar geometry still uses each array's own location.
# Keep the same value across weather sources so a source comparison differs only by dataset.
weatherCellDeg = 0.1

# Pause between API requests (seconds) by source (NSRDB: max 1 request/second; PVGIS: no daily cap, kept polite)
apiPauseSec = {'nsrdb': 1.5, 'pvgis': 0.2}

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Model settings

# Number of operating years to report (opYr 1 = instYr)
nYears = 25

# Total DC system losses (PVWatts default: soiling 2, shading 3, snow 0, mismatch 2, wiring 2, connections 0.5, LID 1.5, nameplate 1, age 0, availability 3 -> 14.08%)
systemLosses = 0.14

# Nominal inverter efficiency (PVWatts default)
invEffNom = 0.96

# Single-axis tracker maximum rotation angle (degrees) and backtracking
trackerMaxAngle = 60
trackerBacktrack = True

# Default orientation where avgAzimuth is missing (degrees from north): fixed-axis faces south; single-axis axis runs north-south (panels track east-west)
defaultFixedAzimuth = 180.0
defaultTrackerAxisAzimuth = 180.0

# Tilt above which fixed-axis values are printed for review (not changed; vertical bifacial systems can be legitimately ~90)
tiltReviewAbove = 60

# Transposition and cell temperature models
transpositionModel = 'perez'
faimanU0, faimanU1 = 25.0, 6.84 # pvlib faiman defaults

# Module temperature coefficient of power (1/C) by modType (PVWatts: standard -0.0037, thin-film -0.0032)
gammaByModType = {'c-si': -0.0037, 'mono-c-si': -0.0037, 'multi-c-si': -0.0040,
                  'cdte': -0.0032, 'cigs': -0.0036, 'a-si': -0.0020, 'thin-film-other': -0.0032, 'thin-film': -0.0032}

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Mount assignment settings

# Mount values modeled directly
mountDirect = {'fixed_axis': 'fixed', 'single_axis': 'single_axis', 'dual_axis': 'dual_axis'}

# Any mount starting with 'mixed' is modeled as fixed-axis (conservative)
mixedMountAs = 'fixed'

# Unknown mounts: 'capCutoff' (derive cutoff from known fixed vs single-axis arrays) or a fixed value in MWDC
unknownMountMethod = 'capCutoff'
unknownCapCutoffManual = None # e.g., 5.0 to override the derived cutoff

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Degradation settings (Jordan et al., 2022)

# Median PLR (fraction per year) by PV temperature zone. Zones <= T3 use T3; zones >= T5 use T5 (Jordan reports T3-T5 only).
plrByTempZone = {3: 0.0048, 4: 0.0078, 5: 0.0088}
plrZoneClip = (3, 5)
plrFleetMedian = 0.0075 # fleet median, used if a zone cannot be assigned

# pvcz zone column (rack-mounted, 1.1 eV activation energy, as in Karin et al.)
pvczZoneCol = 'T_equiv_rack_1p1eV_zone'

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Data conventions

# GM-SEUS no-data value and the columns used here
noData = -9999
arrayIDcol = 'arrayID'
gmseusCols = ['arrayID', 'Source', 'sourceConf', 'latitude', 'longitude', 'instYr', 'capMWDC', 'capMWAC',
              'modType', 'mount', 'tilt', 'avgAzimuth', 'GCR2']

# Helper Functions

In [ ]:
# A function to get TMY weather for one weather cell, cached to parquet. Returns an hourly dataframe with ghi, dni, dhi, temp_air, wind_speed (and albedo if available), with a tz-aware index forced to a single year.
def getTmyWeather(lat, lon):

    # Check cache first
    cacheFile = os.path.join(weatherCache_path, 'tmy_' + weatherSource + '_' + str(lat) + '_' + str(lon) + '.parquet')
    if os.path.exists(cacheFile):
        return pd.read_parquet(cacheFile)

    # Download TMY from the selected source
    if weatherSource == 'nsrdb':
        wx, meta = pvlib.iotools.get_nsrdb_psm4_tmy(lat, lon, nlrApiKey, nlrEmail, year='tmy', map_variables=True)
    else:
        wx = pvlib.iotools.get_pvgis_tmy(lat, lon, map_variables=True)[0]
        wx.index = wx.index.tz_convert('Etc/GMT' + '{:+d}'.format(-int(round(lon / 15)))) # UTC -> local standard time (matches NSRDB and Cambium timestamp_local)

    # TMY months come from different years; force one non-leap year so the index is continuous
    wx.index = wx.index.map(lambda t: t.replace(year=2001))
    wx = wx.sort_index()

    # Keep needed columns and save
    keepCols = [c for c in ['ghi', 'dni', 'dhi', 'temp_air', 'wind_speed', 'albedo'] if c in wx.columns]
    wx = wx[keepCols]
    wx.to_parquet(cacheFile)
    time.sleep(apiPauseSec[weatherSource])
    return wx

# A function to model hourly AC energy (MWh) for one array with a PVWatts-style pvlib chain
def simulateHourlyMWh(wx, row):

    # Solar position for the weather timestamps at the array location
    solPos = pvlib.solarposition.get_solarposition(wx.index, row['latitude'], row['longitude'])

    # Surface orientation by mount: fixed, single-axis (backtracking), or dual-axis
    if row['pvMount'] == 'single_axis':
        trk = pvlib.tracking.singleaxis(solPos['apparent_zenith'], solPos['azimuth'], axis_tilt=0, axis_azimuth=row['pvAxisAz'],
                                        max_angle=trackerMaxAngle, backtrack=trackerBacktrack, gcr=row['pvGcr'])
        surfTilt, surfAz = trk['surface_tilt'].fillna(0), trk['surface_azimuth'].fillna(row['pvAxisAz'])
    elif row['pvMount'] == 'dual_axis':
        surfTilt, surfAz = solPos['apparent_zenith'].clip(0, 90), solPos['azimuth']
    else:
        surfTilt, surfAz = row['pvTilt'], row['pvAzimuth']

    # Plane-of-array irradiance (Perez transposition needs extraterrestrial DNI and airmass)
    dniExtra = pvlib.irradiance.get_extra_radiation(wx.index)
    airmass = pvlib.atmosphere.get_relative_airmass(solPos['apparent_zenith'])
    albedo = wx['albedo'] if 'albedo' in wx.columns else 0.2
    poa = pvlib.irradiance.get_total_irradiance(surfTilt, surfAz, solPos['apparent_zenith'], solPos['azimuth'],
                                                wx['dni'], wx['ghi'], wx['dhi'], dni_extra=dniExtra, airmass=airmass,
                                                albedo=albedo, model=transpositionModel)
    poaGlobal = poa['poa_global'].fillna(0).clip(lower=0)

    # Cell temperature (Faiman) and DC power (PVWatts) with system losses
    cellTemp = pvlib.temperature.faiman(poaGlobal, wx['temp_air'], wx['wind_speed'], u0=faimanU0, u1=faimanU1)
    pdc = pvlib.pvsystem.pvwatts_dc(poaGlobal, cellTemp, pdc0=row['capMWDC'] * 1e6, gamma_pdc=row['pvGamma']) * (1 - systemLosses)

    # AC power (PVWatts inverter) clipped at AC capacity; hourly W -> MWh
    pacMax = row['pvCapMWAC'] * 1e6
    pac = pvlib.inverter.pvwatts(pdc, pdc0=pacMax / invEffNom, eta_inv_nom=invEffNom).clip(lower=0, upper=pacMax)
    return pac / 1e6

# A function to collapse hourly energy into a normalized 12x24 (month x hour) shape, flattened to 288 values that sum to 1
def to12x24Shape(hourlyMWh):
    shape = hourlyMWh.groupby([hourlyMWh.index.month, hourlyMWh.index.hour]).sum()
    shape = shape.reindex(pd.MultiIndex.from_product([range(1, 13), range(24)]), fill_value=0.0)
    return (shape / shape.sum()).values

# Column names for the flattened 12x24 shape
shapeCols = ['m' + str(m).zfill(2) + 'h' + str(h).zfill(2) for m in range(1, 13) for h in range(24)]

# Load and Prepare GM-SEUS Arrays

In [ ]:
# Call GM-SEUS arrays and keep only columns used for generation modeling (geometry not needed; centroid latitude/longitude are attributes)
gmseusArrays = gpd.read_file(gmseusArraysPath)
genArrays = pd.DataFrame(gmseusArrays[gmseusCols]).copy()

# Replace no-data values with NaN for modeling
genArrays = genArrays.replace(noData, np.nan)

# Standardize text fields
genArrays['mount'] = genArrays['mount'].str.lower()
genArrays['modType'] = genArrays['modType'].str.lower()

# Flag arrays that will not be modeled (CSP or missing capacity / location / installation year)
genArrays['pvModeled'] = ~((genArrays['modType'] == 'csp') | genArrays['capMWDC'].isna() | genArrays['latitude'].isna() | genArrays['instYr'].isna())

# Print counts
print('Total arrays:', len(genArrays))
print('Arrays modeled with pvlib:', genArrays['pvModeled'].sum())
print('Arrays not modeled (CSP or missing inputs):', (~genArrays['pvModeled']).sum())
print(genArrays['mount'].value_counts())

# Assign Model Inputs
## Mount Technology
Known mounts are used directly, `mixed*` mounts are modeled as fixed-axis, and `unknown` mounts are assigned by a capacity cutoff. The cutoff is chosen from arrays with known fixed-axis vs single-axis mounts as the capacity (in log space) that maximizes balanced accuracy, i.e., the mean of the fixed-axis and single-axis recall. Balanced accuracy is used rather than overall accuracy because we do not know the fixed vs single-axis mix among unknown arrays, and overall accuracy would favor the more common fixed-axis class.

In [ ]:
# Get arrays with known fixed-axis or single-axis mounts and their log capacity
knownMount = genArrays[genArrays['mount'].isin(['fixed_axis', 'single_axis']) & genArrays['pvModeled']].copy()
knownMount['isSingle'] = (knownMount['mount'] == 'single_axis').astype(int)
knownMount['logCap'] = np.log10(knownMount['capMWDC'].clip(lower=1e-4))

# Scan candidate cutoffs and compute balanced accuracy (single-axis if capacity >= cutoff)
candidateCutoffs = np.round(10 ** np.arange(-1.5, 2.51, 0.01), 4)
cutoffScores = []
for cutoff in candidateCutoffs:
    predSingle = knownMount['capMWDC'] >= cutoff
    recallSingle = (predSingle & (knownMount['isSingle'] == 1)).sum() / (knownMount['isSingle'] == 1).sum()
    recallFixed = (~predSingle & (knownMount['isSingle'] == 0)).sum() / (knownMount['isSingle'] == 0).sum()
    cutoffScores.append([cutoff, recallSingle, recallFixed, (recallSingle + recallFixed) / 2])
cutoffScores = pd.DataFrame(cutoffScores, columns=['cutoffMWDC', 'recallSingle', 'recallFixed', 'balancedAcc'])

# Select the best cutoff (or the manual override)
bestCutoff = cutoffScores.loc[cutoffScores['balancedAcc'].idxmax()]
unknownCapCutoff = unknownCapCutoffManual if unknownCapCutoffManual is not None else bestCutoff['cutoffMWDC']
print('Derived cutoff (MWDC):', bestCutoff['cutoffMWDC'], '| balanced accuracy:', round(bestCutoff['balancedAcc'], 3),
      '| single-axis recall:', round(bestCutoff['recallSingle'], 3), '| fixed recall:', round(bestCutoff['recallFixed'], 3))
print('Cutoff used (MWDC):', unknownCapCutoff)

# Diagnostic: balanced accuracy of the cutoff by installation era (trackers became common in utility-scale after ~2015)
knownMount['era'] = pd.cut(knownMount['instYr'], bins=[0, 2014, 2019, 3000], labels=['<=2014', '2015-2019', '>=2020'])
for era, eraDF in knownMount.groupby('era', observed=True):
    predSingle = eraDF['capMWDC'] >= unknownCapCutoff
    rS = (predSingle & (eraDF['isSingle'] == 1)).sum() / max((eraDF['isSingle'] == 1).sum(), 1)
    rF = (~predSingle & (eraDF['isSingle'] == 0)).sum() / max((eraDF['isSingle'] == 0).sum(), 1)
    print('Era', era, '| n =', len(eraDF), '| single share:', round(eraDF['isSingle'].mean(), 2), '| balanced accuracy:', round((rS + rF) / 2, 3))

In [ ]:
# Assign modeled mount (pvMount) and its provenance (pvMountSource)
genArrays['pvMount'] = genArrays['mount'].map(mountDirect)
genArrays['pvMountSource'] = np.where(genArrays['pvMount'].notna(), 'gmseus', None)

# Mixed mounts -> fixed-axis
isMixed = genArrays['mount'].str.startswith('mixed')
genArrays.loc[isMixed, 'pvMount'] = mixedMountAs
genArrays.loc[isMixed, 'pvMountSource'] = 'mixedAssumedFixed'

# Unknown mounts -> capacity cutoff
isUnknown = genArrays['pvMount'].isna()
genArrays.loc[isUnknown, 'pvMount'] = np.where(genArrays.loc[isUnknown, 'capMWDC'] >= unknownCapCutoff, 'single_axis', 'fixed')
genArrays.loc[isUnknown, 'pvMountSource'] = 'unknownCapCutoff'

# Print modeled mount counts and capacity share by provenance
print(pd.crosstab(genArrays['pvMountSource'], genArrays['pvMount']))
print((genArrays.groupby('pvMountSource')['capMWDC'].sum() / genArrays['capMWDC'].sum()).round(3))

## Orientation, Ground Cover Ratio, DC:AC Ratio, and Temperature Coefficient

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Fixed-axis tilt

# Use tilt as-is (already gap-filled with tiltEst in script8)
genArrays['pvTilt'] = genArrays['tilt']
genArrays['pvTiltSource'] = np.where(genArrays['tilt'].notna(), 'gmseus', None)

# Arrays newly assigned fixed-axis (from unknown) have no tilt: fit tilt ~ latitude on fixed-axis arrays with tilt and fill
needTilt = (genArrays['pvMount'] == 'fixed') & genArrays['pvTilt'].isna()
fitDF = genArrays[(genArrays['pvMount'] == 'fixed') & genArrays['pvTilt'].notna()]
tiltSlope, tiltIntercept = np.polyfit(fitDF['latitude'], fitDF['pvTilt'], 1)
genArrays.loc[needTilt, 'pvTilt'] = tiltSlope * genArrays.loc[needTilt, 'latitude'] + tiltIntercept
genArrays.loc[needTilt, 'pvTiltSource'] = 'latitudeRegression'
print('Tilt fill regression: tilt =', round(tiltSlope, 3), 'x latitude +', round(tiltIntercept, 2), '| fitted n =', len(fitDF), '| filled n =', needTilt.sum())

# Print fixed-axis arrays with very steep tilt for review (kept as-is)
print('Fixed-axis arrays with tilt >', tiltReviewAbove, ':', ((genArrays['pvMount'] == 'fixed') & (genArrays['pvTilt'] > tiltReviewAbove)).sum())

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Azimuth (fixed) and tracker axis azimuth (single-axis)

# Fixed-axis: avgAzimuth is the panel-row face direction; missing -> due south
genArrays['pvAzimuth'] = genArrays['avgAzimuth'].fillna(defaultFixedAzimuth)
genArrays['pvAzimuthSource'] = np.where(genArrays['avgAzimuth'].notna(), 'gmseus', 'defaultOrientation')

# Single-axis: avgAzimuth is the long-edge direction = tracker axis; missing -> north-south axis.
# Fold to the south-pointing half (e.g., 0 -> 180, 10 -> 190, 175 -> 175); pvlib treats axis azimuth a and a+180 identically for a horizontal axis.
axisAz = genArrays['avgAzimuth'].fillna(defaultTrackerAxisAzimuth) % 180
genArrays['pvAxisAz'] = np.where(axisAz < 90, axisAz + 180, axisAz)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Ground cover ratio (GCR2) for tracker backtracking

# GCR2 = row width / row pitch, the GCR definition pvlib uses (GCR1 is a packing factor and is not used)
genArrays['pvGcr'] = genArrays['GCR2']

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ AC capacity (inverter clipping) and temperature coefficient

genArrays['pvCapMWAC'] = genArrays['capMWAC']
genArrays['pvGamma'] = genArrays['modType'].map(gammaByModType)

# Checks: DC:AC ratio range, and any modType without a temperature coefficient
print('DC:AC ratio summary:'); print((genArrays['capMWDC'] / genArrays['capMWAC']).describe(percentiles=[0.01, 0.5, 0.99]).round(2))
print('modType values without gamma:', genArrays.loc[genArrays['pvGamma'].isna() & genArrays['pvModeled'], 'modType'].unique())
print(genArrays.loc[genArrays['pvMount'] == 'fixed', 'pvTiltSource'].value_counts())
print(genArrays['pvAzimuthSource'].value_counts())

## PV Temperature Zone and Performance Loss Rate (Degradation)

In [ ]:
# Call pvcz global climate zone grid and build a nearest-neighbor tree on lat/lon
pvczData = pvcz.get_pvcz_data()
pvczTree = cKDTree(np.column_stack([pvczData['lat'].values, pvczData['lon'].values]))

# Find the nearest pvcz grid point for each array and assign the temperature zone
_, nearestIdx = pvczTree.query(np.column_stack([genArrays['latitude'].fillna(0).values, genArrays['longitude'].fillna(0).values]))
genArrays['pvTempZone'] = pvczData[pvczZoneCol].values[nearestIdx].astype(float)

# Assign PLR by zone (zones outside T3-T5 clipped to the nearest reported zone), fleet median if no zone
genArrays['plr'] = genArrays['pvTempZone'].clip(*plrZoneClip).map(plrByTempZone).fillna(plrFleetMedian)

# Print zone and PLR counts
print(genArrays['pvTempZone'].value_counts().sort_index())
print(genArrays['plr'].value_counts())

# Model Generation
## Assign Weather Cells and Download TMY

In [ ]:
# Snap arrays to weather cells
genArrays['wxLat'] = (np.round(genArrays['latitude'] / weatherCellDeg) * weatherCellDeg).round(3)
genArrays['wxLon'] = (np.round(genArrays['longitude'] / weatherCellDeg) * weatherCellDeg).round(3)
genArrays['wxCell'] = genArrays['wxLat'].astype(str) + '_' + genArrays['wxLon'].astype(str)

# Get unique weather cells for modeled arrays
wxCells = genArrays.loc[genArrays['pvModeled'], ['wxCell', 'wxLat', 'wxLon']].drop_duplicates('wxCell').reset_index(drop=True)
print('Modeled arrays:', genArrays['pvModeled'].sum(), '| unique weather cells:', len(wxCells))

In [ ]:
# This cell downloads one TMY per weather cell (cached; safe to re-run after interruption)

# Loop through weather cells and download (or load cached) TMY
failedCells = []
for index, cell in wxCells.iterrows():
    try:
        getTmyWeather(cell['wxLat'], cell['wxLon'])
    except Exception as e:
        failedCells.append([cell['wxCell'], str(e)[:200]])

    # Print percent progress every 5%
    if (index + 1) % max(int(len(wxCells) / 20), 1) == 0:
        print('Progress:', round((index + 1) / len(wxCells) * 100), '%')

# Print failed cells (re-run cell to retry)
print('Failed weather cells:', len(failedCells))

## Simulate TMY Generation for Each Array

In [ ]:
# This cell simulates one TMY year per array, grouped by weather cell. Results are checkpointed so the cell can resume.

# Load checkpoint if it exists
if os.path.exists(genCheckpoint_path):
    genResults = pd.read_parquet(genCheckpoint_path)
    doneIDs = set(genResults[arrayIDcol])
else:
    genResults = pd.DataFrame()
    doneIDs = set()

# Get modeled arrays not yet simulated
toModel = genArrays[genArrays['pvModeled'] & ~genArrays[arrayIDcol].isin(doneIDs)]
cellGroups = list(toModel.groupby('wxCell'))

# Loop through weather cells, simulate each array in the cell, and store annual MWh and 12x24 shape
newRows = []
for i, (wxCell, cellArrays) in enumerate(cellGroups):

    # Load cached TMY for the cell (skip simulation if download failed)
    try:
        wx = getTmyWeather(cellArrays['wxLat'].iloc[0], cellArrays['wxLon'].iloc[0])
    except Exception:
        wx = None

    # Simulate each array in the cell
    if wx is not None:
        for _, row in cellArrays.iterrows():
            hourlyMWh = simulateHourlyMWh(wx, row)
            newRows.append([row[arrayIDcol], hourlyMWh.sum()] + list(to12x24Shape(hourlyMWh)))

    # Checkpoint every 500 cells and at the end
    if ((i + 1) % 500 == 0) or (i + 1 == len(cellGroups)):
        genResults = pd.concat([genResults, pd.DataFrame(newRows, columns=[arrayIDcol, 'MWhTmy'] + shapeCols)], ignore_index=True)
        genResults.to_parquet(genCheckpoint_path)
        newRows = []
        print('Progress:', round((i + 1) / len(cellGroups) * 100), '%')

print('Arrays simulated:', len(genResults))

## Check Capacity Factors
Annual DC capacity factor = MWhTmy / (capMWDC x 8760). Typical US ground-mounted values are roughly 0.12-0.22 for fixed-axis and 0.16-0.27 for single-axis (higher in the Southwest). Values outside `cfFlagRange` are flagged for review, not removed.

In [ ]:
# Join results to model inputs and compute DC capacity factor
genArrays = genArrays.drop(columns=['MWhTmy'], errors='ignore').merge(genResults[[arrayIDcol, 'MWhTmy']], on=arrayIDcol, how='left')
genArrays['cfDC'] = genArrays['MWhTmy'] / (genArrays['capMWDC'] * 8760)

# Summarize by modeled mount
print(genArrays.groupby('pvMount')['cfDC'].describe(percentiles=[0.05, 0.5, 0.95]).round(3))

# Flag outliers
cfFlagRange = (0.08, 0.35)
genArrays['cfFlag'] = ~genArrays['cfDC'].between(*cfFlagRange) & genArrays['cfDC'].notna()
print('Arrays with CF outside', cfFlagRange, ':', genArrays['cfFlag'].sum())

# Apply Degradation and Build Output Tables

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Long table: one row per array and operating year

# Repeat each modeled array nYears times and assign operating year (opYr) and calendar year (calYr)
genLong = genArrays.loc[genArrays['MWhTmy'].notna(), [arrayIDcol, 'instYr', 'plr', 'MWhTmy']].copy()
genLong = genLong.loc[genLong.index.repeat(nYears)].reset_index(drop=True)
genLong['opYr'] = np.tile(np.arange(1, nYears + 1), int(len(genLong) / nYears))
genLong['calYr'] = (genLong['instYr'] + genLong['opYr'] - 1).astype(int)

# Linear degradation: MWh = MWhTmy x (1 - PLR x (opYr - 1))
genLong['MWh'] = genLong['MWhTmy'] * (1 - genLong['plr'] * (genLong['opYr'] - 1))
genLong = genLong[[arrayIDcol, 'instYr', 'opYr', 'calYr', 'MWh']]

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Wide table: genYr1 ... genYrN for every GM-SEUS array (non-modeled = -9999)

genWide = genLong.pivot(index=arrayIDcol, columns='opYr', values='MWh')
genWide.columns = ['genYr' + str(c) for c in genWide.columns]
genWide = genArrays[[arrayIDcol]].merge(genWide.reset_index(), on=arrayIDcol, how='left').fillna(noData)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Checks

# Conservation check: genYr1 equals MWhTmy, and every modeled array has nYears rows
assert np.allclose(genLong[genLong['opYr'] == 1]['MWh'].values, genArrays.loc[genArrays['MWhTmy'].notna(), 'MWhTmy'].values)
assert (genLong.groupby(arrayIDcol).size() == nYears).all()
print('Total genYr1 (TWh):', round(genLong[genLong['opYr'] == 1]['MWh'].sum() / 1e6, 2))
print('Total genYr' + str(nYears) + ' (TWh):', round(genLong[genLong['opYr'] == nYears]['MWh'].sum() / 1e6, 2))

# Export

In [ ]:
# Export wide annual generation (GM-SEUS release), long annual generation, 12x24 shapes, and model inputs
genWide.round(2).to_csv(genWide_path, index=False)
genLong.to_parquet(genLong_path)
genResults[[arrayIDcol] + shapeCols].to_parquet(genShape_path)

# Model input columns (with provenance) for auditing
inputCols = [arrayIDcol, 'Source', 'sourceConf', 'pvModeled', 'mount', 'pvMount', 'pvMountSource', 'pvTilt', 'pvTiltSource',
             'pvAzimuth', 'pvAzimuthSource', 'pvAxisAz', 'pvGcr', 'capMWDC', 'pvCapMWAC', 'modType', 'pvGamma',
             'pvTempZone', 'plr', 'wxCell', 'MWhTmy', 'cfDC', 'cfFlag']
genArrays['weatherSource'] = weatherSource
inputCols = inputCols + ['weatherSource']
genArrays[inputCols].to_csv(genInputs_path, index=False)
print('Exported to:', generation_path)